# Module C, sequence 2: saving and loading (exercises)

Replace each `...` by what its comment asks, then run the cell. The steps follow the TP 2 of the handout of the module C. The corrected notebook is in the branch `correction-module-c`.

In [ ]:
import numpy as np
from pathlib import Path
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data.data import Axis, DataRaw, DataToExport
from pymodaq_data.h5modules.data_saving import DataToExportSaver, DataSaverLoader, DataLoader

DATA = Path('../data')  # a pathlib.Path: the operator / joins path parts (DATA / 'file.h5')

# the objects of the sequence 1
wavelength = Axis('wavelength', 'nm', data=np.linspace(500, 560, 121), index=0)
signal = 3 * gauss1D(wavelength.get_data(), 528, 4) + 0.1
line = DataRaw('line', data=[signal], axes=[wavelength], labels=['photodiode'], units='V')
line.errors = [np.full(121, 0.05)]
mesure = DataToExport('mesure', data=[line])

## 1. Write the file

Then open `line.h5` in H5Browser (`h5browser` in a terminal): plot the line and read its attributes.

In [ ]:
path = Path('line.h5')
with DataToExportSaver(..., new_file=...) as saver:  # the path of the file; True to overwrite it if it exists
    ...  # save `mesure` in the group '/RawData' (hint: add_data)
    for node in saver.h5saver.walk_nodes('/'):  # all the nodes of the file: where are the data, the axis, the errors?
        print(node)
# the file is closed when leaving the `with` block

## 2. Read it back and compare

In [ ]:
with ... as loader:  # open the file at `path` for reading (hint: DataLoader, imported above; the file is closed at the end of the block)
    loaded = ...  # load everything under '/RawData', as a DataToExport (hint: loader.load_all)
back = ...  # the data named 'line' (hint: get_data_from_name)
print(...)  # is it equal to `line` (hint: ==)? Print also its axes and its errors

## 2 bis. One data alone

`DataSaverLoader` saves one data object (a `DataWithAxes`), `DataToExportSaver` a whole `DataToExport`. Compare the
nodes with the ones of the part 1.

In [ ]:
path_one = Path('line_alone.h5')
with DataSaverLoader(path_one, new_file=True) as saver:
    ...  # save `line` in the group '/RawData' (hint: saver.add_data)
with DataLoader(path_one) as loader:
    for node in loader.walk_nodes('/'):
        print(node)
    alone = ...  # load the data node '/RawData/Data00' (hint: loader.load_data)
print(alone == line)

## 3. Read the continuous recording of the DAQ_Viewer (enlargeable arrays)

Use your own recording (the Continuous Save button of the toolbar of the DAQ_Viewer of the spectrometer), or the file of the `data` folder.

In [ ]:
with DataLoader(DATA / 'continuous_spectra.h5') as loader:  # or the path of your own recording
    ...  # print the nodes of the file and find the enlargeable array, EnlData (hint: loader.walk_nodes('/'))
    spectra = ...  # load this array with its navigation axis (hint: load_data, load_all=True)
print(spectra.shape[0], 'spectra recorded, shape', spectra.shape)
timestamps = ...  # the axis of the dimension 0 (hint: get_axis_from_index)
print(timestamps.label, timestamps.units, timestamps.get_data()[:3])

## 4. Mean and standard error of the series

In [ ]:
mean_spectrum = ...  # the mean over the acquisitions, with its standard error in `errors` (hint: spectra.mean_with_errors)
wavelength = ...  # the values of the signal axis, the dimension 1 (hint: get_axis_from_index, get_data)
i_peak = int(np.argmax(mean_spectrum.data[0]))
print(wavelength[i_peak], mean_spectrum.data[0][i_peak], mean_spectrum.errors[0][i_peak])

## Control question 2

Why does the file of the continuous recording not have the same layout as a snapshot file, and what would we lose
by saving each spectrum in a separate file?